# Failure Analysis

Use this notebook to compare BM25 and SBERT failure-detail outputs with pandas.

In [1]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
ROOT = Path.cwd().parent.parent
BM25_FAILURE_DETAILS = ROOT / "artifacts/eval/bm25_failure_details.json"
SBERT_FAILURE_DETAILS = ROOT / "artifacts/eval/sbert_failure_details.json"

In [6]:
def load_failure_details(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def failure_records_to_frame(details: dict, model_name: str) -> pd.DataFrame:
    frame = pd.json_normalize(details["failure_records"])
    frame["model"] = model_name
    frame["source_model_type"] = details.get("source_model_type")
    frame["source_model_version"] = details.get("source_model_version")
    return frame

In [7]:
bm25_details = load_failure_details(BM25_FAILURE_DETAILS)
sbert_details = load_failure_details(SBERT_FAILURE_DETAILS)

bm25_df = failure_records_to_frame(bm25_details, "bm25")
sbert_df = failure_records_to_frame(sbert_details, "sbert")

failures_df = pd.concat([bm25_df, sbert_df], ignore_index=True)
failures_df.head()

,query,score_differential,pos_doc_ratio,neg_doc_ratio,top_negative.id,top_negative.score,top_negative.matched_terms,top_negative.text,first_positive.id,first_positive.score,first_positive.matched_terms,first_positive.text,model,source_model_type,source_model_version
0,who presides over a senate trial trial after a...,30.946320,0.499127,0.327014,5316990,61.081412,"[[a, 0.40354243955724456, 1], [over, 3.0308099...",Who presides over the impeachment trial of a U...,7290412,30.135092,"[[over, 3.030809948891667, 2], [president, 4.7...",This has occurred only twice: 1 Chief Justice ...,bm25,None,None
1,what is the difference in a c corp or s corp,28.720518,0.688451,0.722873,7839830,43.189133,"[[a, 0.40354243955724456, 1], [c, 3.9140437468...",Difference Between C corp and S corp. 1 In C c...,6419354,14.468615,"[[a, 0.40354243955724456, 2], [c, 3.9140437468...",S Corporations. The main difference between a ...,bm25,None,None
2,what is wintv,28.260086,0.998253,0.602394,7849422,29.187943,"[[is, 0.6115663638289429, 2], [what, 2.7895546...",What is WinTV? Computer illiterate here:) I fo...,7849430,0.927857,"[[is, 0.6115663638289429, 3]]",WIN Television. WIN Television is an Australia...,bm25,None,None
3,what is trivora,27.916881,0.533549,0.550760,7219465,28.906016,"[[is, 0.6115663638289429, 2], [trivora, 13.743...",Here is the some steps to help you to save mon...,7219457,0.989135,"[[is, 0.6115663638289429, 2]]",TrivoraÂ® (levonorgestrel/ethinyl estradiol) i...,bm25,None,None
4,what is the difference between a c-corp and a ...,27.170750,0.688451,0.533549,8140501,44.591921,"[[and, 0.31820338284203753, 7], [between, 3.05...",Share it with your friends/family. 1 Differenc...,6419354,17.421171,"[[a, 0.40354243955724456, 2], [and, 0.31820338...",S Corporations. The main difference between a ...,bm25,None,None


In [9]:
numeric_columns = [
    "score_differential"
]

failures_df.groupby("model")[numeric_columns].describe()

score_differential                                                    \
                   count      mean       std       min       25%       50%   
model                                                                        
bm25               666.0  5.947290  5.348229  0.011587  2.067620  4.612230   
sbert              311.0  0.080472  0.070551  0.000390  0.028404  0.062007   

                            
            75%        max  
model                       
bm25   8.212231  30.946320  
sbert  0.112780   0.428881

In [30]:
# score_differential, top_negative.score, first_positive.score, join on query?
same_failures = bm25_df[
    ['query', 'score_differential', 'top_negative.score', 'first_positive.score', 'top_negative.text', 'first_positive.text']
].merge(
    sbert_df[
        ['query', 'score_differential', 'top_negative.score', 'first_positive.score', 'top_negative.text', 'first_positive.text']
    ],
    on='query',
    suffixes=('_bm25', '_sbert')
)
curRow = same_failures.iloc[3]

display(curRow['query'])
print()
display(curRow['top_negative.text_bm25'])
print()
display(curRow['first_positive.text_bm25'])
print()
print()

display(curRow['top_negative.text_sbert'])
print()
display(curRow['first_positive.text_sbert'])

'how long does pink eye last bacterial'

'How long does pink eye last? How long does pink eye last from cum? How long can the redness last in pink eye? How long does it take to catch pink eye? How long does an eye infection last? How long does an eye strain usually last? How long does ringworm live on surfaces? How long does hiv live on surfaces? How long does pink eye germs last on stuffed animals?'

'The viral pink eye can last up to three weeks but usually goes away within days. When the eyes look and feel normal then the contagion is gone. Allergic pink eye can be caused by cosmetics, contact cleaning solutions, and pollen.'

'Pink eye also can be caused by bacterial conjunctivitis, which even with treatment such as prescription antibiotic eye drops can last up to a month or longer. However, with this type of pink eye, people should no longer be contagious 24 hours after antibiotic treatment begins.'

'The viral pink eye can last up to three weeks but usually goes away within days. When the eyes look and feel normal then the contagion is gone. Allergic pink eye can be caused by cosmetics, contact cleaning solutions, and pollen.'

Next steps:

- Add query-id-based comparison once both failure outputs include query ids.
- Explode matched-term lists into a separate frame.
- Compare BM25-only failures against SBERT-only failures.